<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/rr_respiratory_mortality/notebooks/07.RR_Respiratory_Mortality_Cluster_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎 Analysing the Cluster Patterns in Epidemiological Context

The Bivariate LISA analysis identifies where spatial co-clustering between predicted asthma mortality and RR of respiratory mortality occurs, but not why. To move from statistical pattern to epidemiological meaning, cluster types are cross-referenced with the underlying environmental drivers that the Random Forest model was trained on — namely $PM_{2.5}$ and $PM_{10}$ concentrations, land-use/land-cover (LULC) transition variables, and so on. This contextualisation step operationalises the classic spatial epidemiology principle that geographic clustering of disease burden is rarely random, but rather reflects the spatial distribution of exposure, vulnerability, and environmental risk. By systematically comparing the environmental profiles of HL, and LH departments, we can assess whether the detected hotspots are consistent with known pathways linking particulate matter exposure, land-use change, and respiratory mortality — and identify which environmental conditions most strongly discriminate high-burden clusters from the national baseline.

## 🤖 Load libraries

The libraries required for the analysis will be loaded

In [ ]:
# dataframe libraries
import pandas as pd

# Import NumPy for numerical and mathematical operations
import numpy as np

# geospatial libraries
import geopandas as gpd

# plot libraries
import mapclassify
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.colors import Normalize, ListedColormap
import seaborn as sns

# other libraries
from IPython.display import Image
import os
from itables import init_notebook_mode, show

In [ ]:
# change directory to work folder (at the begining docker container enter into /home/jovyan/)
%cd work

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

## ⟳ Load and transform the environmental dataset


Before any contextual interpretation is possible, the environmental dataset must be loaded, filtered to the 170 departments included in the LISA analysis (i.e., where asthma mortality actually occurred in 2022) , and arranged so its structure matches the LISA results GeoDataFrame. This prepocessing step ensures that the join key (IDDPTO) is present a in both datasets, that only the analytically relevant variables are retained, and the spatial and tabular structure of the environmental GeoDataFrame is consistent with the LISA output.

In [ ]:
# Load gpkg file with environmental data
gdf_am = gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# Load gpkg file with LISA results
gdf_ca_rrrm_pj = gpd.read_file('pdt/rr_respiratory_mortality/data/gpkg/gdf_ca_rrrm_pj_cluster.gpkg')

In [ ]:
# # keep columns IDPROV PROV IDDPTO DPTO and all ending in 2022
# columns_to_keep = ['IDPROV', 'PROV', 'IDDPTO', 'DPTO', 'geometry'] + [col for col in gdf_am.columns if col.endswith('2022')]

# keep columns IDPROV PROV IDDPTO DPTO and all ending in 2022
columns_to_keep = ['IDPROV', 'IDDPTO', 'geometry'] + [col for col in gdf_am.columns if col.endswith('2022')]

In [ ]:
# filter gdf using columns_to_keep
gdf_2022= gdf_am[columns_to_keep]

In [ ]:
# verify columns
gdf_2022.columns

In [ ]:
# drop 'A_2022' 'C_2022', 'AMR_2022'
gdf_2022 = gdf_2022.drop(columns=['A_2022', 'C_2022', "AMR_2022"])

In [ ]:
# verify columns
gdf_2022.columns

In [ ]:
# from gdf_2022 select the same IDDPTO in  gpkg file with LISA results
gdf_2022_170_deps = gdf_2022[gdf_2022['IDDPTO'].isin(gdf_ca_rrrm_pj['IDDPTO'])]

In [ ]:
# reproject gdf_2022_170_deps to EPSG:5347
gdf_2022_170_deps = gdf_2022_170_deps.to_crs(5347)

## 🕵 Inspect the Environmental Dataset

Before merging and interpreting environmental data with the LISA results, an inspection of the dataset is essential to ensure data integrity. This includes verifying that all 170 departments are present, variable types are suitable for numeric analysis, the coordinate reference system is consistent with the LISA GeoDataFrame, and  no missing values are present.

In [ ]:
# Shape: confirm 170 rows and expected number of columns
print(f"Shape of environmental GeoDataFrame : {gdf_2022_170_deps.shape}")
print(f"  → Rows (departments) : {gdf_2022_170_deps.shape[0]}")
print(f"  → Columns            : {gdf_2022_170_deps.shape[1]}")

In [ ]:
# Column names and data types + non-null counts
# .info() prints dtype and the count of non-null values for every column.
# Any column showing < 170 non-null values has missing data.
print("----Column Info----")
gdf_2022_170_deps.info()
print()

In [ ]:
# Coordinate Reference System
# Must match gdf (EPSG:5347) to avoid misalignment during any spatial join.
print(f"CRS of environmental GDF : {gdf_2022_170_deps.crs}")
print(f"CRS of LISA results GDF  : {gdf_ca_rrrm_pj.crs}")
print(f"CRS match                : {gdf_2022_170_deps.crs == gdf_ca_rrrm_pj.crs}")

In [ ]:
# Descriptive statistics for all numeric environmental variables
# This produces the baseline (mean)
init_notebook_mode(all_interactive=True)
env_cols = ['PM25_2022', 'PM10_2022', 'NBA_2022', 'PD_2022', 'NAGRT_2022', 'NNWVT_2022', 'NBUT_2022']
display(gdf_2022_170_deps[env_cols].describe().round(4))

In [ ]:
# Missing value check per column
print("── Missing Values per Column ────────────────────────────────────────────")
print(gdf_2022_170_deps[env_cols].isnull().sum().to_string())
print()

In [ ]:
# Confirm IDDPTO is present and has no duplicate entries
# Duplicates in the join key would cause a many-to-many merge and inflate rows.
print(f"IDDPTO present in columns : {'IDDPTO' in gdf_2022_170_deps.columns}")
print(f"Duplicate IDDPTO values   : {gdf_2022_170_deps['IDDPTO'].duplicated().sum()}")
print(f"Unique IDDPTO count       : {gdf_2022_170_deps['IDDPTO'].nunique()}")

In [ ]:
# Check if gdf_ca_rrrm_pj and  gdf_2022_170_deps contain the exact same set of IDs
np.array_equal(gdf_ca_rrrm_pj['IDDPTO'].sort_values().values,
               gdf_2022_170_deps['IDDPTO'].sort_values().values)

##🧬 Merge Environmental Variables with LISA Cluster Results

In this section, we will be joining the environmental dataset to the LISA results GeoDataFrame on the shared key IDDPTO to create a single table in which each department's cluster classification (HL, LH, Not Significant) is paired with its full environmental data.

In [ ]:
# Select only the columns needed from the environmental GDF for the merge
# We exclude 'geometry' from the right-side table to avoid a geometry conflict;
# the LISA GDF already carries the geometry for all 170 depts.
env_cols_to_merge = ['IDDPTO', 'IDPROV',
                     'PM25_2022', 'PM10_2022', 'NBA_2022', 'PD_2022',
                     'NAGRT_2022', 'NNWVT_2022', 'NBUT_2022']

In [ ]:
# Perform a left merge on IDDPTO
# left_on / right_on both point to 'IDDPTO' — the shared department identifier.
# how='left' guarantees every LISA department is retained; unmatched rows → NaN.
gdf_merged = gdf_ca_rrrm_pj.merge(
    gdf_2022_170_deps[env_cols_to_merge],  # right table: environmental variables only
    on='IDDPTO',                            # join key present in both tables
    how='left'                              # retain all rows from the LISA GDF
)

In [ ]:
# Confirm row count is unchanged (must remain 170)
print(f"Rows after merge : {len(gdf_merged)}  (expected: 170)")

In [ ]:
# Post-merge NaN check for environmental columns
# Any NaN here means an IDDPTO in LISA results GDF had no match in the environmental table.
env_cols = ['PM25_2022', 'PM10_2022', 'NBA_2022', 'PD_2022',
            'NAGRT_2022', 'NNWVT_2022', 'NBUT_2022']

print("── Post-merge NaN check (environmental columns) ─────────────────────────")
print(gdf_merged[env_cols].isnull().sum().to_string())

In [ ]:
# Preview the merged record for HL departments (high predicted asthma mortality/low RR of respiratory mortality)
hl_ids = gdf_merged[gdf_merged['CLUSTERS'] == 'HL']['IDDPTO'].tolist()

init_notebook_mode(all_interactive=True)
display(
    gdf_merged[gdf_merged['IDDPTO'].isin(hl_ids)][
        ['IDDPTO', 'DPTO', 'PROV', 'CLUSTERS',
         'AMR_PRED_2022', 'RRRM_2022_C'] + env_cols
    ].sort_values('AMR_PRED_2022', ascending=False)
)

In [ ]:
# Preview the merged record for LH departments (low predicted asthma mortality/high RR of respiratory mortality)
lh_ids = gdf_merged[gdf_merged['CLUSTERS'] == 'LH']['IDDPTO'].tolist()

init_notebook_mode(all_interactive=True)
display(
    gdf_merged[gdf_merged['IDDPTO'].isin(lh_ids)][
        ['IDDPTO', 'DPTO', 'PROV', 'CLUSTERS',
         'AMR_PRED_2022', 'RRRM_2022_C'] + env_cols
    ].sort_values('AMR_PRED_2022', ascending=False))

## ⛏️ Extract environmental data by cluster type and evaluation against reference baselines at the department level


In this section, we extract environmental data for departments classified as HL, and LH clusters. These data are then compared with the national mean values of environmental variables to assess whether cluster types differ from the national average (170 departments).



In [ ]:
# Define the list of environmental variables
env_cols = ['PM25_2022', 'PM10_2022', 'NBA_2022', 'PD_2022',
            'NAGRT_2022', 'NNWVT_2022', 'NBUT_2022']

In [ ]:
# Compute national mean of environmental variables.
# National mean computed across 170 departments where asthma mortality occured in 2022
national_mean = gdf_merged[env_cols].mean()

In [ ]:
# visualize national mean of environmental variables
init_notebook_mode(all_interactive=True)
national_mean

In [ ]:
# All cluster mean: mean across all clustered departments with p < 0.05
# (HL + LH ) — represents the "spatially structured" subset
all_cluster_mean = gdf_merged[gdf_merged['CLUSTERS'] != "NO CLUSTER"][env_cols].mean()

In [ ]:
#visualize all cluster mean
init_notebook_mode(all_interactive=True)
all_cluster_mean

In [ ]:
# Extract data of HL department
hl = gdf_merged[gdf_merged['CLUSTERS'] == 'HL'].copy()

In [ ]:
# Extract data of LH department
lh = gdf_merged[gdf_merged['CLUSTERS'] == 'LH'].copy()

In [ ]:
# Compute cluster means of nvironmental variables  for HL cluster
hl_mean = hl[env_cols].mean()

In [ ]:
# Compute cluster means of nvironmental variables  for LH cluster
lh_mean = lh[env_cols].mean()

In [ ]:
# Compute ratio table (cluster type mean / national mean)
# A ratio > 1 indicates the cluster exceeds the national average for that variable.
# A ratio < 1 indicates it falls below. This highlights discriminatory variables.
comparison_ratio = pd.DataFrame({
    'HL / National'           : hl_mean   / national_mean,
    'LH / National'           : lh_mean   / national_mean,
    'All cluster / National'  : all_cluster_mean  / national_mean
}).T.round(4)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
comparison_ratio

HL clusters were characterized by a markedly higher mean burned area (3.29 times the national value) and by greater agricultural and livestock (1.73) and natural wooded vegetation (1.35) transition, with very low population density (0.02) and lower PM2.5 (0.68). LH clusters showed the opposite profile: population density (1.45), PM2.5 (1.65) and built-up transition (2.63) exceeded national values, while burned area was almost null (0.01). PM10 was close to the national value in both cluster types (0.98 to 0.99) and did not discriminate between them. Overall, HL clusters resemble sparsely populated departments with intense land-cover change and fire activity, whereas LH clusters resemble densely populated, urbanizing departments with higher particulate exposure. These ecological contrasts are descriptive and do not allow causal or individual-level inference.

### 🌌 Mapping Environmental Variables for HL/LH Context

In [ ]:
# open gdf data
gdf = gpd.read_file("pdt/rr_respiratory_mortality/data/gpkg/gdf_rrrm.gpkg")

In [ ]:
# select all NaN rows for gdf
gdf_nan = gdf[gdf.isna().any(axis=1)]

In [ ]:
# from gdf_nan select IDDPTO and geometry
gdf_nan = gdf_nan[['IDDPTO', 'geometry']]

In [ ]:
# reproject gdf_nan
gdf_nan_pj = gdf_nan.to_crs(5347)

In [ ]:
init_notebook_mode(all_interactive=True)
gdf_merged.head()

In [ ]:
# LH = lightblue → low mortality, high neighbouring attributable rate
# HL = orange    → high mortality, low neighbouring attributable rate
# NO CLUSTER = lightgrey
cluster_colors = {
    #'HH':'#d7191c',# crimson red
    'LH':'#4682B4',# light blue
    'HL':'#fdae61',# orange
    'NO CLUSTER': '#949494'# light grey
}

In [ ]:
# Map cluster labels to colours in the GeoDataFrame
# Every department row receives a hex colour based on its cluster label
gdf_merged['COLOR'] = gdf_merged['CLUSTERS'].map(cluster_colors)

In [ ]:
# Prepare HL, and LH boundary layers for overlay
# These will be drawn as thick coloured outlines on every environmental map.
gdf_hl_boundary = gdf_merged[gdf_merged['CLUSTERS'] == 'HL']
gdf_lh_boundary = gdf_merged[gdf_merged['CLUSTERS'] == 'LH']

In [ ]:
# Define the environmental variables to map and their display settings ─
map_vars = {
    'PM25_2022' : {
        'label' : 'PM2.5 (µg/m³)',
    },
    'PM10_2022' : {
        'label' : 'PM10 (µg/m³)',
    },
    'NBA_2022'  : {
        'label' : 'Burned Area\n(norm. per 1000 km²)'
    },
    'PD_2022'   : {
        'label' : 'Population Density\n(inhab./km²)'
    },
    'NAGRT_2022': {
        'label' : 'Agricultural Transition\n(norm. per 1000 km²)'
    },
    'NNWVT_2022': {
        'label' : 'Natural Vegetation Loss\n(norm. per 1000 km²)',
    },
       'NBUT_2022' : {
        'label' : 'Built-up Transition\n(norm. per 1000 km²)',
    }
  }

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 22)) # 2 rows, 4 columns


fig.suptitle(
    'Bivariate LISA Clusters and Environmental variables\n'
    'Argentina — 2022',
    fontsize=14, fontweight='bold', y=1.01
)

# Panel 1: Bivariate LISA cluster map (reference)
ax0 = axes[0, 0]

# Plot the 170 analysed departments coloured by LISA cluster
gdf_merged.plot(
    ax        = ax0,
    color     = gdf_merged['COLOR'],    # 'color' column set in Step 5
    edgecolor = 'white',
    linewidth = 0.3
)

# Overlay NaN departments as hollow polygons
gdf_nan_pj.plot(
    ax        = ax0,
    color     = 'none',
    edgecolor = 'black',
    linewidth = 0.15
)

# Add hollowed rectangles around each cluster type on ax0 with specific styling
for cluster_type in ['HL', 'LH']:
    cluster_data = gdf_merged[gdf_merged['CLUSTERS'] == cluster_type]
    if not cluster_data.empty:
        minx, miny, maxx, maxy = cluster_data.union_all().bounds

        edge_color = 'black'
        linewidth_rect = 1.5
        linestyle = '' # Default, will be set below
        buffer_factor = 0 # Default no buffer

        if cluster_type == 'HL':
            linestyle = ':' # Dotted black line
        elif cluster_type == 'LH':
            linestyle = '-' # Solid black line

        # Apply buffer if specified
        if buffer_factor > 0:
            width = maxx - minx
            height = maxy - miny
            minx -= width * buffer_factor
            miny -= height * buffer_factor
            maxx += width * buffer_factor
            maxy += height * buffer_factor

        rect_patch = mpatches.Rectangle((minx, miny), maxx - minx, maxy - miny,
                                        facecolor='none', edgecolor=edge_color, linewidth=linewidth_rect, linestyle=linestyle, zorder=5)
        ax0.add_patch(rect_patch)

# Build a compact legend for the LISA map (excluding HH)
lisa_patches = [
    mpatches.Patch(facecolor='#fdae61', label='HL'),
    mpatches.Patch(facecolor='#abd9e9', label='LH'),
    mpatches.Patch(facecolor='#949494', label='Not Significant'),
    mpatches.Patch(facecolor='none',    edgecolor='black', linewidth=0.8, label='No data')
]
ax0.legend(handles=lisa_patches, fontsize=7, loc='lower left',
           title='LISA Clusters', title_fontsize=7)

ax0.set_title('Bivariate LISA Cluster Map',
              fontsize=10, fontweight='bold')
ax0.set_axis_off()

# Environmental choropleth maps
# Flatten axes and skip index 0 (LISA map); include axes[1,3] to show all 7 variables
ax_list = [axes[0,1], axes[0,2], axes[0,3], axes[1,0], axes[1,1], axes[1,2], axes[1,3]]

for ax, (col, settings) in zip(ax_list, map_vars.items()):
    #print(col)

    # Quantile classification (5 classes)
    classifier = mapclassify.Quantiles(gdf_merged[col].dropna(), k=5)

    # Plot the choropleth
    gdf_merged.plot(
        column     = col,
        ax        = ax,
        scheme    = 'quantiles',
        k         = 5,
        cmap      = "viridis",
        edgecolor = 'white',
        linewidth = 0.2,
        legend    = False
    )

    # Overlay NaN departments as hollow polygons
    gdf_nan_pj.plot(
        ax        = ax,
        color     = 'none',
        edgecolor = 'black',
        linewidth = 0.12
    )

    # Add hollowed rectangles around each cluster type with specific styling
    for cluster_type in ['HL', 'LH']:
        cluster_data = gdf_merged[gdf_merged['CLUSTERS'] == cluster_type]
        if not cluster_data.empty:
            minx, miny, maxx, maxy = cluster_data.union_all().bounds

            edge_color = 'black'
            linewidth_rect = 1.5
            linestyle = '' # Default, will be set below
            buffer_factor = 0 # Default no buffer

            if cluster_type == 'HL':
                linestyle = ':' # Dotted black line
            elif cluster_type == 'LH':
                linestyle = '-' # Solid black line

            # Apply buffer if specified
            if buffer_factor > 0:
                width = maxx - minx
                height = maxy - miny
                minx -= width * buffer_factor
                miny -= height * buffer_factor
                maxx += width * buffer_factor
                maxy += height * buffer_factor

            rect_patch = mpatches.Rectangle((minx, miny), maxx - minx, maxy - miny,
                                            facecolor='none', edgecolor=edge_color, linewidth=linewidth_rect, linestyle=linestyle, zorder=5)
            ax.add_patch(rect_patch)

    # Colorbar
    sm = ScalarMappable(
        cmap  = "viridis",
        norm  = Normalize(
            vmin = gdf_merged[col].min(),
            vmax = gdf_merged[col].max()
        )
    )
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax     = divider.append_axes('right', size='4%', pad=0.05)
    plt.colorbar(sm, cax=cax, label=settings['label'])

    # Update Legend for HL and LH boundary overlays (excluding HH)
    hl_line = mlines.Line2D([], [], color='black', linewidth=1.5, linestyle=':', label='HL boundary (dotted)')
    lh_line = mlines.Line2D([], [], color='black', linewidth=1.5, linestyle='-', label='LH boundary (solid)')
    ax.legend(handles=[hl_line, lh_line], fontsize=7, loc='lower left')

    ax.set_title(settings['label'], fontsize=10, fontweight='bold')
    ax.set_axis_off()

plt.tight_layout()
plt.show()

### 🚩 Conclussion

The statistical comparison an the visual inspection through the choropleth maps of environmental variables by cluster type against the national mean (across 170 departments) reveals distinct exposure profiles for each bivariate LISA cluster.


* **HL departments** (high predicted mortality, low RR) display a rural–agricultural profile, with burned areas more than three times the national mean (3.29), increased agricultural (1.73 times national mean) and natural wooded vegetation (1.35 times national mean) transitions, and relative low population density and urban expansion. This pattern suggests that **burned areas, together with agricultural expansion and vegetation losses, are the main environmental drivers in this hotspot**.

* **LH departments** (low predicted mortality, high RR) have moderate to elevated PM₂.₅ (1.65 times national mean), population density (1.45 times national mean), and built-up transitions (2.65), suggesting a ** cluster structure influenced by PM₂.₅ exposure, urban expansion, and population density.**

Overall, the results indicate that spatial patterns of respiratory mortality are not driven by a single environmental variable but by cluster-specific combinations of urban density, air pollution, and land-use changes.